
# ANTARES hybrid loci/alerts visualization

Visualization notebook (previous name `antares_hybrid_visualization_v1`) for the clean hybrid ANTARES download in:

```text
antares_data_hybrid/
    loci/
    alerts/
```

This follows the visualization workflow we discussed earlier, adapted to the current v2/v1 hybrid Parquet schema.

Main plots:

1. Overall locus sky distribution.
2. Overall alert sky distribution.
3. Sky distribution split by observing month.
4. Alert counts by month.
5. Band distribution.
6. Science-image magnitude distribution.
7. Alerts per locus.
8. Basic Gaia / Milliquas match summaries.

## Science-image magnitude

The preferred magnitude is derived from the **diaObject mean science-image flux** for the relevant band:

```text
lsst_diaObject_<band>_scienceFluxMean
```

with AB magnitude

\[
m_{\rm AB} = 31.4 - 2.5\log_{10}(f_{\rm nJy}).
\]

When the alert band can be identified, the notebook uses the matching band's diaObject mean flux. It also extracts the alert-level:

```text
lsst_diaSource_scienceFlux
```

as a fallback / comparison.

The notebook prints the actual LSST property keys and the fraction of alerts for which each magnitude can be constructed, so it is easy to verify the field names against the downloaded data.


In [ ]:

from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
})

OUTDIR = Path("antares_data_hybrid")
LOCUS_DIR = OUTDIR / "loci"
ALERT_DIR = OUTDIR / "alerts"

MJD_MIN = 61187.0
AB_NJY_ZEROPOINT = 31.4


## Load Parquet datasets

In [ ]:

loci_ds = ds.dataset(
    LOCUS_DIR,
    format="parquet",
)

alerts_ds = ds.dataset(
    ALERT_DIR,
    format="parquet",
)

loci = loci_ds.to_table().to_pandas()

alerts = alerts_ds.to_table(
    columns=[
        "locus_id",
        "alert_id",
        "mjd",
        "alert_properties",
    ]
).to_pandas()

print(f"Locus rows:       {len(loci):,}")
print(f"Unique locus IDs: {loci['locus_id'].nunique():,}")
print()

print(f"Alert rows:       {len(alerts):,}")
print(f"Unique alert IDs: {alerts['alert_id'].nunique():,}")
print()

print(f"Minimum MJD:      {alerts['mjd'].min():.6f}")
print(f"Maximum MJD:      {alerts['mjd'].max():.6f}")
print(f"All MJD >= cut:   {alerts['mjd'].min() >= MJD_MIN}")


## Parse LSST alert properties

In [ ]:

def parse_json_or_empty(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    try:
        return json.loads(x)
    except Exception:
        return {}


# Inspect a sample of keys before extracting anything.
sample_keys = set()

for x in alerts["alert_properties"].head(5000):
    sample_keys.update(
        parse_json_or_empty(x).keys()
    )

lsst_keys = sorted(
    k for k in sample_keys
    if "lsst" in k.lower()
)

print(f"LSST-like property keys found in sample: {len(lsst_keys)}")
for key in lsst_keys:
    print(key)



### Extract band and science-image fluxes

For each alert the code tries to determine:

- alert band;
- `lsst_diaSource_scienceFlux`;
- all available `lsst_diaObject_<band>_scienceFluxMean` values;
- preferred diaObject mean science flux in the alert's own band.


In [ ]:

BANDS = ["u", "g", "r", "i", "z", "y"]


def first_present(props, keys):
    for key in keys:
        value = props.get(key)
        if value is not None:
            return value
    return None


def to_float_or_nan(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def get_alert_band(props):
    value = first_present(
        props,
        [
            "lsst_diaSource_band",
            "lsst_diaSource_filterName",
            "lsst_diaSource_filter",
            "band",
        ],
    )

    if value is None:
        return None

    value = str(value).strip().lower()

    for band in BANDS:
        if value == band or value.endswith(f"_{band}"):
            return band

    return value


def get_diaobject_mean_fluxes(props):
    # Return {band: scienceFluxMean} for all available LSST diaObject bands.
    out = {}

    for band in BANDS:
        candidates = [
            f"lsst_diaObject_{band}_scienceFluxMean",
            f"lsst_diaObject_{band}ScienceFluxMean",
        ]

        value = first_present(
            props,
            candidates,
        )

        if value is not None:
            out[band] = to_float_or_nan(value)

    pattern = re.compile(
        r"lsst_diaObject_([ugrizy])_?scienceFluxMean$",
        re.IGNORECASE,
    )

    for key, value in props.items():
        m = pattern.match(key)
        if m:
            band = m.group(1).lower()
            out[band] = to_float_or_nan(value)

    return out


def flux_njy_to_mag(flux):
    flux = np.asarray(flux, dtype=float)

    mag = np.full(
        flux.shape,
        np.nan,
        dtype=float,
    )

    good = np.isfinite(flux) & (flux > 0)

    mag[good] = (
        AB_NJY_ZEROPOINT
        - 2.5 * np.log10(flux[good])
    )

    return mag


records = []

for x in alerts["alert_properties"]:
    props = parse_json_or_empty(x)

    band = get_alert_band(props)

    dia_source_flux = to_float_or_nan(
        first_present(
            props,
            [
                "lsst_diaSource_scienceFlux",
                "diaSource_scienceFlux",
            ],
        )
    )

    mean_fluxes = get_diaobject_mean_fluxes(
        props
    )

    preferred_mean_flux = np.nan

    if band in mean_fluxes:
        preferred_mean_flux = mean_fluxes[band]

    records.append({
        "band": band,
        "diaSource_scienceFlux": dia_source_flux,
        "diaObject_mean_scienceFlux": preferred_mean_flux,
        **{
            f"diaObject_{b}_scienceFluxMean":
                mean_fluxes.get(b, np.nan)
            for b in BANDS
        },
    })


flux_info = pd.DataFrame(
    records,
    index=alerts.index,
)

alerts = pd.concat(
    [
        alerts,
        flux_info,
    ],
    axis=1,
)

alerts["diaSource_scienceMag"] = flux_njy_to_mag(
    alerts["diaSource_scienceFlux"].values
)

alerts["diaObject_mean_scienceMag"] = flux_njy_to_mag(
    alerts["diaObject_mean_scienceFlux"].values
)

for band in BANDS:
    alerts[f"diaObject_{band}_scienceMagMean"] = flux_njy_to_mag(
        alerts[f"diaObject_{band}_scienceFluxMean"].values
    )


n = len(alerts)

print(
    "Alert-level diaSource scienceFlux available:",
    f"{alerts['diaSource_scienceFlux'].notna().sum():,}/{n:,}",
)

print(
    "Band-matched diaObject scienceFluxMean available:",
    f"{alerts['diaObject_mean_scienceFlux'].notna().sum():,}/{n:,}",
)

print()
print("Band counts:")
print(
    alerts["band"]
    .fillna("unknown")
    .value_counts()
)


## Add dates and locus coordinates to alerts

In [ ]:

# Unix epoch corresponds to MJD 40587.
alerts["datetime"] = pd.to_datetime(
    alerts["mjd"] - 40587.0,
    unit="D",
    origin="unix",
    utc=True,
)

alerts["month"] = (
    alerts["datetime"]
    .dt.tz_localize(None)
    .dt.to_period("M")
    .astype(str)
)

coord = loci[
    [
        "locus_id",
        "ra",
        "dec",
    ]
].drop_duplicates(
    "locus_id"
)

alerts = alerts.merge(
    coord,
    on="locus_id",
    how="left",
)

print(
    "Earliest:",
    alerts["datetime"].min().strftime("%Y-%m-%d %H:%M:%S UTC"),
)

print(
    "Latest:  ",
    alerts["datetime"].max().strftime("%Y-%m-%d %H:%M:%S UTC"),
)

print(
    "Alerts missing coordinates:",
    alerts["ra"].isna().sum(),
)


## Overall locus sky distribution

In [ ]:

fig, ax = plt.subplots(
    figsize=(5.5, 3.6)
)

ax.scatter(
    loci["ra"],
    loci["dec"],
    s=2,
    alpha=0.35,
    rasterized=True,
)

ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title(
    f"Locus sky distribution ({len(loci):,} loci)"
)

ax.set_xlim(360, 0)

fig.tight_layout()
plt.show()


## Overall alert sky distribution

In [ ]:

fig, ax = plt.subplots(
    figsize=(5.5, 3.6)
)

good = (
    alerts["ra"].notna()
    & alerts["dec"].notna()
)

ax.scatter(
    alerts.loc[good, "ra"],
    alerts.loc[good, "dec"],
    s=1,
    alpha=0.18,
    rasterized=True,
)

ax.set_xlabel("RA [deg]")
ax.set_ylabel("Dec [deg]")
ax.set_title(
    f"Alert sky distribution ({good.sum():,} alerts)"
)

ax.set_xlim(360, 0)

fig.tight_layout()
plt.show()



## Monthly sky distribution

To avoid giving a locus extra visual weight simply because it generated many alerts in the same month, each panel plots **unique loci active in that month**.


In [ ]:

months = sorted(
    alerts["month"].dropna().unique()
)

n_month = len(months)
ncol = min(3, max(1, n_month))
nrow = math.ceil(n_month / ncol)

fig, axes = plt.subplots(
    nrow,
    ncol,
    figsize=(
        4.0 * ncol,
        3.0 * nrow,
    ),
    squeeze=False,
    sharex=True,
    sharey=True,
)

dec_min = np.nanmin(loci["dec"])
dec_max = np.nanmax(loci["dec"])
dec_pad = 0.03 * (dec_max - dec_min)

for ax, month in zip(
    axes.flat,
    months,
):
    sub = (
        alerts.loc[
            alerts["month"] == month,
            [
                "locus_id",
                "ra",
                "dec",
            ],
        ]
        .dropna(
            subset=["ra", "dec"]
        )
        .drop_duplicates(
            "locus_id"
        )
    )

    ax.scatter(
        sub["ra"],
        sub["dec"],
        s=2,
        alpha=0.4,
        rasterized=True,
    )

    ax.set_title(
        f"{month}  (N={len(sub):,})"
    )

    ax.set_xlim(360, 0)
    ax.set_ylim(
        dec_min - dec_pad,
        dec_max + dec_pad,
    )

    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("Dec [deg]")


for ax in axes.flat[n_month:]:
    ax.set_visible(False)


fig.suptitle(
    "Monthly sky distribution of active loci",
    y=1.01,
)

fig.tight_layout()
plt.show()


## Alert counts by month

In [ ]:

monthly_counts = (
    alerts["month"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(5.5, 3.5)
)

bars = ax.bar(
    monthly_counts.index,
    monthly_counts.values,
)

for bar, count in zip(
    bars,
    monthly_counts.values,
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_xlabel("Month")
ax.set_ylabel("Number of alerts")
ax.set_title("LSST alerts by month")

ax.tick_params(
    axis="x",
    rotation=45,
)

fig.tight_layout()
plt.show()


## Band distribution

In [ ]:

band_counts = (
    alerts["band"]
    .fillna("unknown")
    .value_counts()
)

order = [
    b for b in BANDS
    if b in band_counts.index
]

order += [
    x for x in band_counts.index
    if x not in order
]

band_counts = band_counts.reindex(
    order
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

bars = ax.bar(
    band_counts.index,
    band_counts.values,
)

for bar, count in zip(
    bars,
    band_counts.values,
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_xlabel("Band")
ax.set_ylabel("Number of alerts")
ax.set_title("Alert band distribution")

fig.tight_layout()
plt.show()



## Science-image magnitude

### Preferred: diaObject mean science magnitude

Because `diaObject_<band>_scienceFluxMean` is an object-level mean and may be repeated in several alerts from the same locus, the plot below first deduplicates by **locus + band**.


In [ ]:

mean_mag_rows = []

for band in BANDS:
    col = f"diaObject_{band}_scienceMagMean"

    sub = alerts[
        [
            "locus_id",
            col,
        ]
    ].dropna(
        subset=[col]
    )

    sub = sub.groupby(
        "locus_id",
        as_index=False,
    )[col].median()

    if len(sub) == 0:
        continue

    tmp = pd.DataFrame({
        "locus_id": sub["locus_id"],
        "band": band,
        "mag": sub[col],
    })

    mean_mag_rows.append(tmp)


if mean_mag_rows:
    object_mag = pd.concat(
        mean_mag_rows,
        ignore_index=True,
    )
else:
    object_mag = pd.DataFrame(
        columns=[
            "locus_id",
            "band",
            "mag",
        ]
    )


print(
    f"Unique locus-band diaObject mean magnitudes: "
    f"{len(object_mag):,}"
)

print(
    object_mag["band"]
    .value_counts()
    .reindex(BANDS)
    .dropna()
)


fig, ax = plt.subplots(
    figsize=(4.5, 3.3)
)

bins = np.arange(
    14,
    25.25,
    0.25,
)

for band in BANDS:
    values = object_mag.loc[
        object_mag["band"] == band,
        "mag",
    ]

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        continue

    ax.hist(
        values,
        bins=bins,
        histtype="step",
        linewidth=1.5,
        label=f"{band} (N={len(values):,})",
        log=True,
    )

ax.set_xlabel("diaObject mean science magnitude [AB]")
ax.set_ylabel("Number of locus-band measurements")
ax.set_title("Science-image mean magnitude")
ax.legend(
    frameon=False,
    fontsize=8,
)

fig.tight_layout()
plt.show()



### Comparison: alert-level diaSource science magnitude

This uses `lsst_diaSource_scienceFlux` and therefore represents individual detections rather than the diaObject mean.


In [ ]:

fig, ax = plt.subplots(
    figsize=(4.5, 3.3)
)

bins = np.arange(
    14,
    25.25,
    0.25,
)

for band in BANDS:
    values = alerts.loc[
        alerts["band"] == band,
        "diaSource_scienceMag",
    ]

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        continue

    ax.hist(
        values,
        bins=bins,
        histtype="step",
        linewidth=1.5,
        label=f"{band} (N={len(values):,})",
        log=True,
    )

ax.set_xlabel("diaSource science magnitude [AB]")
ax.set_ylabel("Number of alerts")
ax.set_title("Alert-level science magnitude")
ax.legend(
    frameon=False,
    fontsize=8,
)

fig.tight_layout()
plt.show()


## Alerts per locus

In [ ]:

alerts_per_locus = (
    alerts
    .groupby("locus_id")
    .size()
)

fig, ax = plt.subplots(
    figsize=(4, 3)
)

bins = np.arange(
    0.5,
    alerts_per_locus.max() + 1.5,
    1,
)

ax.hist(
    alerts_per_locus.values,
    bins=bins,
    histtype="step",
    linewidth=1.5,
    log=True,
)

ax.set_xlabel("Number of retained alerts per locus")
ax.set_ylabel("Number of loci")
ax.set_title("Alerts per locus")

fig.tight_layout()
plt.show()

print(alerts_per_locus.describe())


## Catalog-match summary

In [ ]:

milliquas_n = int(
    loci["milliquas_match"]
    .fillna(False)
    .sum()
)

gaia_qso_n = 0

if "gaia_in_qso_candidates" in loci.columns:
    gaia_qso_n = sum(
        any(
            v is True
            for v in (
                x if isinstance(x, (list, np.ndarray))
                else []
            )
        )
        for x in loci["gaia_in_qso_candidates"]
    )

print(f"Total loci:                  {len(loci):,}")
print(f"Milliquas matched:           {milliquas_n:,}")
print(f"Gaia QSO-candidate flag:     {gaia_qso_n:,}")


## Quick consistency checks

In [ ]:

print(
    "Duplicate locus rows:",
    len(loci) - loci["locus_id"].nunique(),
)

print(
    "Duplicate alert rows:",
    len(alerts) - alerts["alert_id"].nunique(),
)

print(
    "All alerts after May 27:",
    alerts["mjd"].min() >= MJD_MIN,
)

print(
    "All alert locus IDs exist:",
    set(alerts["locus_id"]).issubset(
        set(loci["locus_id"])
    ),
)

print(
    "Missing alert coordinates:",
    alerts["ra"].isna().sum(),
)


In [ ]:
alerts_per_locus = (
    alerts.groupby("locus_id")
    .size()
)

two_alert_ids = (
    alerts_per_locus[
        alerts_per_locus == 2
    ]
    .index
)

check = loci[
    loci["locus_id"].isin(two_alert_ids)
][
    [
        "locus_id",
        "num_tagged_alerts",
        "max_score",
    ]
]

print(check.head(20))

print()
print(
    check["num_tagged_alerts"]
    .value_counts()
    .sort_index()
)

In [ ]:
alerts_per_locus = (
    alerts.groupby("locus_id")
    .size()
)

two_alert_ids = (
    alerts_per_locus[
        alerts_per_locus == 2
    ]
    .index
)

check = loci.loc[
    loci["locus_id"].isin(two_alert_ids),
    [
        "locus_id",
        "num_tagged_alerts",
        "max_score",
    ],
].copy()

print("Loci with only 2 saved alerts:", len(check))
print()

print(
    check["num_tagged_alerts"]
    .value_counts()
    .sort_index()
)

print()
print(check.head(20))

In [ ]:
import requests
from urllib.parse import urljoin

V2_BASE_URL = "https://api.antares.noirlab.edu/v2/"


def get_all_v2_data(url, params=None):
    rows = []

    while url:

        r = requests.get(
            url,
            params=params,
            timeout=60,
        )

        r.raise_for_status()

        response = r.json()

        rows.extend(
            response.get("data", [])
        )

        next_url = (
            response
            .get("links", {})
            .get("next")
        )

        url = (
            urljoin(url, next_url)
            if next_url
            else None
        )

        # next URL already contains pagination info
        params = None

    return rows

In [ ]:
test_id = check.iloc[0]["locus_id"]

raw = get_all_v2_data(
    V2_BASE_URL + f"loci/{test_id}/alerts",
    params={
        "limit": 100,
        "elasticsearch_query": json.dumps({}),
    },
)

print("locus:", test_id)
print("num_tagged_alerts:",
      check.iloc[0]["num_tagged_alerts"])
print("raw alerts returned:", len(raw))

for item in raw:
    a = item.get("alert", item)
    print(
        a.get("id"),
        a.get("mjd"),
    )

In [ ]:
for locus_id in check["locus_id"]:

    raw = get_all_v2_data(
        V2_BASE_URL + f"loci/{locus_id}/alerts",
        params={
            "limit": 100,
            "elasticsearch_query": json.dumps({}),
        },
    )

    raw_info = []

    for item in raw:
        a = item.get("alert", item)

        raw_info.append({
            "alert_id": a.get("id"),
            "mjd": a.get("mjd"),
        })

    raw_df = pd.DataFrame(raw_info)

    saved_ids = set(
        alerts.loc[
            alerts["locus_id"] == locus_id,
            "alert_id",
        ]
    )

    print()
    print("locus:", locus_id)
    print(
        "num_tagged_alerts:",
        loci.loc[
            loci["locus_id"] == locus_id,
            "num_tagged_alerts",
        ].iloc[0],
    )
    print("raw alerts returned:", len(raw_df))
    print(
        "unique raw alert IDs:",
        raw_df["alert_id"].nunique(),
    )
    print("saved alerts:", len(saved_ids))

    print(raw_df)